In [1]:
import ast
import json
import re
import os
import time
import hashlib
from pathlib import Path
from datetime import datetime
from collections import Counter
import asyncio

import pandas as pd
import numpy as np
from tqdm.asyncio import tqdm
import aiohttp
import aiofiles
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import imagehash

# FIX: This patch allows asyncio to work in environments that already have a running event loop (like Jupyter)
import nest_asyncio
nest_asyncio.apply()

# --- SETUP & CONFIGURATION ---
print("─" * 80)
print("▶ [STEP 1] Initializing Configuration for OPTIMIZED Pipeline...")
print("─" * 80)

# --- File & Directory Paths ---
RAW_CSV_PATH = Path('MM-Food-100K.csv')
OUTPUT_DIR = Path('preprocessed_data')
IMAGES_DIR = OUTPUT_DIR / 'images'

OUTPUT_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

# --- Image Processing Settings ---
CONCURRENT_REQUESTS = 150
MAX_IMAGE_DIM = 512
JPEG_QUALITY = 85
MIN_IMAGE_RES = (224, 224)

# --- Data Column Settings ---
NUTRIENT_KEYS = ['calories_kcal', 'fat_g', 'protein_g', 'carbohydrate_g']
FINAL_COLUMNS = [
    'dish_name', 'image_filepath', 'submission_date', 'food_type', 'cooking_method',
    'ingredients', 'total_grams'
] + NUTRIENT_KEYS + [f'{key}_per_100g' for key in NUTRIENT_KEYS] + [
    'image_status', 'is_duplicate_image'
]

print(f"Configuration complete. Output will be in: '{OUTPUT_DIR.resolve()}'")
print(f"Using up to {CONCURRENT_REQUESTS} concurrent image downloads.")


────────────────────────────────────────────────────────────────────────────────
▶ [STEP 1] Initializing Configuration for OPTIMIZED Pipeline...
────────────────────────────────────────────────────────────────────────────────
Configuration complete. Output will be in: 'C:\Users\Aaditya\Desktop\NutriSage\preprocessed_data'
Using up to 150 concurrent image downloads.


In [2]:
def safe_json_parse(s):
    if pd.isna(s): return {}
    try: return json.loads(s)
    except:
        try: return ast.literal_eval(s)
        except: return {}

def clean_text(s):
    if not isinstance(s, str): return ''
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s\-']", ' ', s)
    s = re.sub(r"\s+", ' ', s).strip()
    return s

def parse_ingredients(s):
    if pd.isna(s): return []
    try:
        parsed_list = ast.literal_eval(s)
        if isinstance(parsed_list, list):
            return sorted(list(set(clean_text(item) for item in parsed_list if item)))
    except: pass
    return sorted(list(set(clean_text(item) for item in str(s).split(',') if item)))

def parse_total_grams(s):
    if pd.isna(s): return np.nan
    total_grams = 0; found_grams = False
    g_matches = re.findall(r'(\d+\.?\d*)\s*g', str(s).lower())
    kg_matches = re.findall(r'(\d+\.?\d*)\s*kg', str(s).lower())
    for g in g_matches: total_grams += float(g); found_grams = True
    for kg in kg_matches: total_grams += float(kg) * 1000; found_grams = True
    return total_grams if found_grams else np.nan

async def download_and_process_image_async(session, data_tuple, semaphore):
    index, url = data_tuple
    async with semaphore:
        if not isinstance(url, str) or not url.startswith('http'):
            return {'index': index, 'status': 'invalid_url', 'phash': None, 'filepath': None}
        filename = f"{hashlib.sha1(url.encode()).hexdigest()}.jpg"
        filepath = IMAGES_DIR / filename
        if filepath.exists():
            try:
                with Image.open(filepath) as img: phash = str(imagehash.phash(img))
                return {'index': index, 'status': 'exists', 'phash': phash, 'filepath': str(filepath)}
            except: pass
        try:
            async with session.get(url, timeout=30) as response:
                response.raise_for_status()
                if 'image' not in response.headers.get('Content-Type', ''):
                    return {'index': index, 'status': 'not_an_image', 'phash': None, 'filepath': None}
                image_data = await response.read()
                with Image.open(BytesIO(image_data)) as img:
                    if img.width < MIN_IMAGE_RES[0] or img.height < MIN_IMAGE_RES[1]:
                        return {'index': index, 'status': 'too_small', 'phash': None, 'filepath': None}
                    phash = str(imagehash.phash(img))
                    img_rgb = img.convert('RGB')
                    if max(img_rgb.size) > MAX_IMAGE_DIM:
                        img_rgb.thumbnail((MAX_IMAGE_DIM, MAX_IMAGE_DIM), Image.Resampling.LANCZOS)
                    buffer = BytesIO()
                    img_rgb.save(buffer, format='JPEG', quality=JPEG_QUALITY)
                    final_image_data = buffer.getvalue()
                async with aiofiles.open(filepath, 'wb') as f: await f.write(final_image_data)
                return {'index': index, 'status': 'ok', 'phash': phash, 'filepath': str(filepath)}
        except Exception:
            return {'index': index, 'status': 'download_error', 'phash': None, 'filepath': None}

async def process_images_main(df):
    image_urls_with_indices = list(df['image_url'].reset_index().itertuples(index=False, name=None))
    tasks, semaphore = [], asyncio.Semaphore(CONCURRENT_REQUESTS)
    async with aiohttp.ClientSession() as session:
        for item in image_urls_with_indices:
            tasks.append(download_and_process_image_async(session, item, semaphore))
        results = await tqdm.gather(*tasks, desc="Downloading images (async)")
    return results

print("Helper functions defined.")


Helper functions defined.


In [3]:
print("─" * 80); print("▶ [STEP 3] Loading Raw Dataset..."); print("─" * 80)

if not RAW_CSV_PATH.exists():
    print(f"ERROR: Dataset not found at '{RAW_CSV_PATH}'. Please ensure the file is in the same directory.")
else:
    df = pd.read_csv(RAW_CSV_PATH)
    print(f"Loaded {len(df)} records.")
    # In a notebook, display() provides richer output than print() for DataFrames
    from IPython.display import display
    display(df.head())


────────────────────────────────────────────────────────────────────────────────
▶ [STEP 3] Loading Raw Dataset...
────────────────────────────────────────────────────────────────────────────────
Loaded 100000 records.


,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt
0,https://file.b18a.io/7843322356500104680_44354...,0.7,0.95,Fried Chicken,Restaurant food,"[""chicken"",""breading"",""oil""]","[""chicken:300g""]","{""fat_g"":25.0,""protein_g"":30.0,""calories_kcal""...",Frying,20250704
1,https://file.b18a.io/7833227147700100732_67487...,0.7,1.00,Pho,Restaurant food,"[""noodles"",""beef"",""basil"",""lime"",""green onions...","[""noodles:200g"",""beef:100g"",""vegetables:50g""]","{""fat_g"":15.0,""protein_g"":25.0,""calories_kcal""...",boiled,20250702
2,https://file.b18a.io/7832600581600103585_26423...,0.8,0.95,Pan-fried Dumplings,Restaurant food,"[""dumplings"",""chili oil"",""soy sauce""]","[""dumplings:300g"",""sauce:50g""]","{""fat_g"":15.0,""protein_g"":20.0,""calories_kcal""...",Pan-frying,20250625
3,https://file.b18a.io/7839056601700101188_98515...,0.7,1.00,Bananas,Raw vegetables and fruits,"[""Bananas""]","[""Bananas: 10 pieces (about 1kg)""]","{""fat_g"":3.0,""protein_g"":12.0,""calories_kcal"":...",Raw,20250718
4,https://file.b18a.io/7837642737500100261_17312...,0.8,0.90,Noodle Stir-Fry,Restaurant food,"[""noodles"",""chicken"",""vegetables"",""sauce""]","[""noodles:300g"",""chicken:100g"",""vegetables:50g""]","{""fat_g"":20.0,""protein_g"":25.0,""calories_kcal""...",stir-fried,20250711


In [4]:
print("─" * 80); print("▶ [STEP 4] Processing Text & Engineering Features..."); print("─" * 80)

# Parse nutrients
nutrients_df = pd.json_normalize(df['nutritional_profile'].apply(safe_json_parse))
for key in NUTRIENT_KEYS: df[key] = pd.to_numeric(nutrients_df.get(key), errors='coerce')

# Clean text fields
df['dish_name_clean'] = df['dish_name'].apply(clean_text)
df['ingredients_clean'] = df['ingredients'].apply(parse_ingredients)
df['cooking_method_clean'] = df['cooking_method'].apply(clean_text)
df['food_type_clean'] = df['food_type'].apply(clean_text)

# Parse date and create features
df['submission_date'] = pd.to_datetime(df['sub_dt'], format='%Y%m%d', errors='coerce')
df['total_grams'] = df['portion_size'].apply(parse_total_grams)
for key in NUTRIENT_KEYS: df[f'{key}_per_100g'] = (df[key] / df['total_grams']) * 100

print("Text processing and feature engineering complete.")
print("DataFrame with new columns:")
from IPython.display import display
display(df[['dish_name_clean', 'ingredients_clean', 'total_grams'] + NUTRIENT_KEYS].head())


────────────────────────────────────────────────────────────────────────────────
▶ [STEP 4] Processing Text & Engineering Features...
────────────────────────────────────────────────────────────────────────────────
Text processing and feature engineering complete.
DataFrame with new columns:


,dish_name_clean,ingredients_clean,total_grams,calories_kcal,fat_g,protein_g,carbohydrate_g
0,fried chicken,"[breading, chicken, oil]",300.0,400,25.0,30.0,15.0
1,pho,"[basil, beef, chili, green onions, lime, noodles]",350.0,450,15.0,25.0,60.0
2,pan-fried dumplings,"[chili oil, dumplings, soy sauce]",350.0,400,15.0,20.0,50.0
3,bananas,[bananas],1000.0,1050,3.0,12.0,270.0
4,noodle stir-fry,"[chicken, noodles, sauce, vegetables]",450.0,600,20.0,25.0,80.0


In [4]:
# This code block defines the necessary asynchronous functions but does not run them yet.

# --- Helper function with efficiency improvements ---
async def download_and_process_image_async(session, data_tuple, semaphore, processed_dir, max_dim, jpeg_quality, min_res):
    """
    Async worker with two key improvements:
    1. Resumability: Skips downloading if the processed file already exists.
    2. Storage Efficiency: Resizes and compresses images before saving.
    """
    index, url = data_tuple
    async with semaphore:
        if not isinstance(url, str) or not url.startswith('http'):
            return {'index': index, 'status': 'invalid_url', 'phash': None, 'filepath': None}

        filename = f"{hashlib.sha1(url.encode()).hexdigest()}.jpg"
        processed_filepath = processed_dir / filename

        # --- EFFICIENCY 1: RESUMABILITY ---
        if processed_filepath.exists():
            try:
                with Image.open(processed_filepath) as img:
                    phash = str(imagehash.phash(img))
                return {'index': index, 'status': 'exists', 'phash': phash, 'filepath': str(processed_filepath)}
            except Exception:
                pass # The existing file might be corrupt, so we'll re-download.

        try:
            async with session.get(url, timeout=45) as response:
                response.raise_for_status()
                if 'image' not in response.headers.get('Content-Type', ''):
                    return {'index': index, 'status': 'not_an_image', 'phash': None, 'filepath': None}
                
                image_data = await response.read()

                with Image.open(BytesIO(image_data)) as img:
                    if img.width < min_res[0] or img.height < min_res[1]:
                        return {'index': index, 'status': 'too_small', 'phash': None, 'filepath': None}

                    phash = str(imagehash.phash(img))
                    img_rgb = img.convert('RGB')

                    # --- EFFICIENCY 2: STORAGE SAVING ---
                    if max(img_rgb.size) > max_dim:
                        img_rgb.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
                    
                    buffer = BytesIO()
                    img_rgb.save(buffer, format='JPEG', quality=jpeg_quality)
                    final_image_data = buffer.getvalue()

                async with aiofiles.open(processed_filepath, 'wb') as f:
                    await f.write(final_image_data)

                return {'index': index, 'status': 'ok', 'phash': phash, 'filepath': str(processed_filepath)}

        except (asyncio.TimeoutError, aiohttp.ClientError):
            return {'index': index, 'status': 'download_error', 'phash': None, 'filepath': None}
        except Exception:
            return {'index': index, 'status': 'processing_error', 'phash': None, 'filepath': None}

# --- Orchestrator function ---
async def process_images_main(df, processed_dir, max_dim, jpeg_quality, min_res, concurrent_requests):
    """Main async function to orchestrate all image downloads, passing config correctly."""
    image_urls_with_indices = list(df['image_url'].reset_index().itertuples(index=False, name=None))
    tasks = []
    semaphore = asyncio.Semaphore(concurrent_requests)

    async with aiohttp.ClientSession() as session:
        for item in image_urls_with_indices:
            tasks.append(download_and_process_image_async(session, item, semaphore, processed_dir, max_dim, jpeg_quality, min_res))
        
        results = await tqdm.gather(
            *tasks,
            desc="Downloading images (async)"
        )
    return results

print("Asynchronous image processing functions are now defined and ready to use.")


Asynchronous image processing functions are now defined and ready to use.


In [7]:
print("─" * 80); print("▶ [STEP 5] Creating a Stratified Sample..."); print("─" * 80)

# --- CONFIGURATION FOR THIS STEP ---
# Set the desired size for our smaller, representative dataset.
TARGET_DATASET_SIZE = 60000

# We need the scikit-learn library for stratified sampling.
# If you don't have it, run: pip install scikit-learn
try:
    from sklearn.model_selection import train_test_split
except ImportError:
    print("Scikit-learn is not installed. Please install it by running: pip install scikit-learn")
    df_to_process = df.sample(n=TARGET_DATASET_SIZE, random_state=42) # Fallback to random sample
else:
    # FIX: Ensure the column for stratification exists before using it.
    # This makes this step self-contained and prevents the KeyError.
    if 'food_type_clean' not in df.columns:
        print("Creating 'food_type_clean' column for stratification...")
        # The 'clean_text' function was defined in a previous cell.
        df['food_type_clean'] = df['food_type'].apply(clean_text)

    # Create the smaller, representative subset if a target size is set
    if TARGET_DATASET_SIZE and TARGET_DATASET_SIZE < len(df):
        print(f"Creating a stratified sample of {TARGET_DATASET_SIZE} images based on 'food_type_clean'...")
        
        # Stratify by 'food_type_clean' to maintain the original distribution of food types
        df_to_process, _ = train_test_split(
            df,
            train_size=TARGET_DATASET_SIZE,
            stratify=df['food_type_clean'],
            random_state=42  # for reproducibility
        )
        print("Stratified sample created successfully.")
        
        # Verify the distribution is preserved
        original_dist = df['food_type_clean'].value_counts(normalize=True) * 100
        sample_dist = df_to_process['food_type_clean'].value_counts(normalize=True) * 100
        
        comparison_df = pd.DataFrame({
            'Original Distribution (%)': original_dist,
            'Sample Distribution (%)': sample_dist
        }).round(2)
        
        print("\nDistribution of 'food_type_clean' is preserved:")
        from IPython.display import display
        display(comparison_df)
        
    else:
        print("TARGET_DATASET_SIZE is not set or is larger than the dataframe. Processing the full dataset.")
        df_to_process = df



────────────────────────────────────────────────────────────────────────────────
▶ [STEP 5] Creating a Stratified Sample...
────────────────────────────────────────────────────────────────────────────────
Creating 'food_type_clean' column for stratification...
Creating a stratified sample of 60000 images based on 'food_type_clean'...
Stratified sample created successfully.

Distribution of 'food_type_clean' is preserved:


,Original Distribution (%),Sample Distribution (%)
food_type_clean,,
homemade food,46.56,46.56
restaurant food,35.46,35.46
raw vegetables and fruits,9.36,9.36
packaged food,8.35,8.35
others,0.27,0.27


In [11]:
# This cell executes the image processing task on the stratified sample.

print("─" * 80); print("▶ [STEP 6] Executing Image Processing on the Sampled Dataset..."); print("─" * 80)

# --- CONFIGURATION FOR THIS STEP ---
IMAGES_PROCESSED_DIR = OUTPUT_DIR / 'images_processed'
IMAGES_PROCESSED_DIR.mkdir(exist_ok=True) # Ensure the directory exists
BATCH_SIZE = 5000
PROGRESS_CSV_PATH = OUTPUT_DIR / f'processing_progress_{TARGET_DATASET_SIZE}.csv'

# --- FIX for KeyError: ---
# Ensure all columns that will be created during processing exist *before* the loop starts.
# This prevents the 'KeyError' on the very first run.
for col in ['status', 'phash', 'filepath', 'is_duplicate_image']:
    if col not in df.columns:
        df[col] = np.nan

# --- FIX: Re-align the sample with the main dataframe to include the new columns ---
df_to_process = df.loc[df_to_process.index]

print(f"\nProcessing a dataset of {len(df_to_process)} images in batches of {BATCH_SIZE}. The script is resumable.")
print(f"Progress will be saved to '{PROGRESS_CSV_PATH}' after each batch.")

# Identify which rows in our target dataframe still need processing.
# This line will now work correctly because the 'status' column is guaranteed to exist.
unprocessed_df = df_to_process[~df_to_process['status'].isin(['ok', 'exists'])]

if unprocessed_df.empty:
    print("\nAll images in the target dataset have already been processed. Nothing to do.")
else:
    print(f"\nFound {len(unprocessed_df)} images remaining to be processed in this run.")
    num_batches = (len(unprocessed_df) + BATCH_SIZE - 1) // BATCH_SIZE
    
    for i in range(num_batches):
        start_index = i * BATCH_SIZE
        end_index = start_index + BATCH_SIZE
        batch_df = unprocessed_df.iloc[start_index:end_index]
        
        print(f"\n--- Processing Batch {i+1}/{num_batches} ({len(batch_df)} images) ---")
        
        image_metadata = await process_images_main(
            batch_df, IMAGES_PROCESSED_DIR, MAX_IMAGE_DIM, JPEG_QUALITY, MIN_IMAGE_RES, CONCURRENT_REQUESTS
        )
        
        if image_metadata:
            img_meta_df = pd.DataFrame([m for m in image_metadata if m is not None]).set_index('index')
            # Update the main dataframe with the results from this batch
            df.update(img_meta_df)
        
        # --- CHECKPOINTING: SAVE PROGRESS AFTER BATCH ---
        try:
            # We save the full 'df' to keep track of all progress over time
            df.to_csv(PROGRESS_CSV_PATH, index=False)
            print(f"--- Batch {i+1} complete. Progress saved. ---")
        except Exception as e:
            print(f"--- WARNING: Could not save progress after batch {i+1}. Error: {e} ---")

# --- Final steps after all processing is done ---
print("\nIdentifying duplicate images within the processed set...")
processed_rows = df.loc[df_to_process.index] # Only consider duplicates within our sample
successful_downloads = processed_rows[processed_rows['status'].isin(['ok', 'exists']) & processed_rows['phash'].notna()].copy()
duplicates = successful_downloads[successful_downloads.duplicated('phash', keep='first')]
df.loc[duplicates.index, 'is_duplicate_image'] = True


# Final save of the completed dataframe
df.to_csv(PROGRESS_CSV_PATH, index=False)

print("\nImage processing is complete.")
print("\nFinal image status distribution for all processed rows:")
from IPython.display import display
display(df['status'].value_counts())



────────────────────────────────────────────────────────────────────────────────
▶ [STEP 6] Executing Image Processing on the Sampled Dataset...
────────────────────────────────────────────────────────────────────────────────

Processing a dataset of 60000 images in batches of 5000. The script is resumable.
Progress will be saved to 'preprocessed_data\processing_progress_60000.csv' after each batch.

Found 60000 images remaining to be processed in this run.

--- Processing Batch 1/12 (5000 images) ---


  warnings.warn(
C:\Users\Aaditya\AppData\Local\Temp\ipykernel_836\3087308974.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.update(img_meta_df)
C:\Users\Aaditya\AppData\Local\Temp\ipykernel_836\3087308974.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.update(img_meta_df)
C:\Users\Aaditya\AppData\Local\Temp\ipykernel_836\3087308974.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.update(img

--- Batch 1 complete. Progress saved. ---

--- Processing Batch 2/12 (5000 images) ---


--- Batch 2 complete. Progress saved. ---

--- Processing Batch 3/12 (5000 images) ---


  warnings.warn(


...

--- Batch 11 complete. Progress saved. ---

--- Processing Batch 12/12 (5000 images) ---


--- Batch 12 complete. Progress saved. ---

Identifying duplicate images within the processed set...


C:\Users\Aaditya\AppData\Local\Temp\ipykernel_836\3087308974.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[duplicates.index, 'is_duplicate_image'] = True



Image processing is complete.

Final image status distribution for all processed rows:


status
download_error      47534
ok                  10996
not_an_image          983
too_small             486
processing_error        1
Name: count, dtype: int64

In [15]:
# This is the final step of the preprocessing pipeline.

print("─" * 80); print("▶ [STEP 7] Assembling the Final Clean Dataset..."); print("─" * 80)

# --- 1. Filter for Successfully Processed Rows ---
# We create a new dataframe containing only the rows where the image was successfully processed ('ok' or 'exists').
# Based on your output, this will be ~11,000 rows.
clean_df = df[df['status'].isin(['ok', 'exists'])].copy()

print(f"Created a final clean dataset with {len(clean_df)} rows, containing only the successfully processed images.")

# --- 2. Select and Rename Final Columns for Clarity ---
# We select only the columns relevant for modeling and give them clean, simple names.

# First, define the final list of columns we want to keep.
final_columns_to_keep = [
    'dish_name_clean',
    'filepath',
    'submission_date',
    'food_type_clean',
    'cooking_method_clean',
    'ingredients_clean',
    'total_grams',
    'is_duplicate_image'
] + NUTRIENT_KEYS + [f'{key}_per_100g' for key in NUTRIENT_KEYS]


# Rename columns for the final dataset
clean_df = clean_df.rename(columns={
    'dish_name_clean': 'dish_name',
    'filepath': 'image_filepath',
    'food_type_clean': 'food_type',
    'cooking_method_clean': 'cooking_method',
    'ingredients_clean': 'ingredients'
})

# Adjust the 'final_columns_to_keep' list to match the new names
final_columns_to_keep = [col.replace('_clean', '') for col in final_columns_to_keep]
final_columns_to_keep[1] = 'image_filepath' # Adjust filepath name

# Filter the DataFrame to keep only our desired columns in a clean order
df_to_save = clean_df[[col for col in final_columns_to_keep if col in clean_df.columns]]


# --- 3. Save the Final Clean CSV ---
# This is the final output file you will use for your project.
final_csv_path = OUTPUT_DIR / 'MM-Food-Cleaned-Final.csv'
df_to_save.to_csv(final_csv_path, index=False)

print(f"\n✅ Final clean dataset saved successfully to: '{final_csv_path}'")
print("\nThis is the file you will load for your EDA and model training.")

# --- 4. Display a Preview of the Final Data ---
print("\nPreview of the final, clean dataset:")
from IPython.display import display
display(df_to_save.head())

print("\n--- PREPROCESSING COMPLETE ---")



────────────────────────────────────────────────────────────────────────────────
▶ [STEP 7] Assembling the Final Clean Dataset...
────────────────────────────────────────────────────────────────────────────────
Created a final clean dataset with 10996 rows, containing only the successfully processed images.

✅ Final clean dataset saved successfully to: 'preprocessed_data\MM-Food-Cleaned-Final.csv'

This is the file you will load for your EDA and model training.

Preview of the final, clean dataset:


,dish_name,image_filepath,submission_date,food_type,food_type,cooking_method,ingredients,total_grams,is_duplicate_image,calories_kcal,fat_g,protein_g,carbohydrate_g,calories_kcal_per_100g,fat_g_per_100g,protein_g_per_100g,carbohydrate_g_per_100g
13,spicy crab,preprocessed_data\images_processed\0a0c19eb67a...,2025-07-05,Restaurant food,restaurant food,stir-fried,"[crab, sauce, spices, vegetables]",450.0,False,500,20.0,40.0,10.0,111.111111,4.444444,8.888889,2.222222
25,steamed snails,preprocessed_data\images_processed\066b7ba6197...,2025-07-01,Homemade food,homemade food,steamed,"[dipping sauce, garlic, herbs, snails]",350.0,False,200,5.0,30.0,10.0,57.142857,1.428571,8.571429,2.857143
31,chicken stirfry,preprocessed_data\images_processed\fecd381a783...,2025-07-18,Homemade food,homemade food,stir-frying,"[bell peppers, chicken, onions, soy sauce]",450.0,False,450,20.0,30.0,25.0,100.000000,4.444444,6.666667,5.555556
61,grilled salmon,preprocessed_data\images_processed\01e3ca00199...,2025-07-19,Restaurant food,restaurant food,Grilling,"[butter sauce, herbs, salmon]",250.0,False,350,20.0,30.0,5.0,140.000000,8.000000,12.000000,2.000000
72,braised pork belly,preprocessed_data\images_processed\34256ac8f85...,2025-07-04,Restaurant food,restaurant food,Braised,"[Green Onion, Pork Belly, Rice, Soy Sauce]",350.0,False,600,30.0,25.0,50.0,171.428571,8.571429,7.142857,14.285714



--- PREPROCESSING COMPLETE ---
